In [ ]:
#0 Montar o Drive e apontar o dataset construído no notebook 01
# O dataset (imagens + metadata.jsonl) é produzido pelo notebook 01_dataset.ipynb.
# Aqui apenas montamos o Drive, definimos o MESMO DATA_DIR e verificamos que está pronto.
from google.colab import drive
drive.mount('/content/drive')

import os, json
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/multimodais/dados'
os.environ['DATA_DIR'] = DATA_DIR          # exportado para a célula de treino usar via "$DATA_DIR"

meta_path = os.path.join(DATA_DIR, 'metadata.jsonl')
assert os.path.exists(meta_path), f'metadata.jsonl não encontrado em {DATA_DIR}. Rode o notebook 01_dataset.ipynb primeiro.'
meta = [json.loads(l)['file_name'] for l in open(meta_path, encoding='utf-8')]
imgs = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(('.jpg', '.png'))]
faltando = [m for m in meta if not os.path.exists(os.path.join(DATA_DIR, m))]
print(f'DATA_DIR = {DATA_DIR}')
print(f'Imagens: {len(imgs)} | entradas metadata: {len(meta)} | faltando: {len(faltando)}')
if faltando:
    print('  faltando:', faltando)
assert imgs and not faltando, 'Dataset incompleto. Rode/complete o notebook 01 antes de treinar.'

In [ ]:
#1 Preparação do ambiente + treino (notebook 02)

# torchao NÃO é necessário para treinar LoRA e vinha quebrado (wheel py3.10 em ambiente py3.12).
!pip uninstall -y diffusers torchao

# Dependências (aspas evitam o bug de redirecionamento do shell em ">=")
!pip -q install "transformers" "accelerate" "datasets" "peft"

# diffusers a partir do código-fonte (necessário para o script de exemplo)
![ -d /content/diffusers ] || git clone --depth 1 https://github.com/huggingface/diffusers
!pip -q install -e /content/diffusers
%cd /content/diffusers/examples/text_to_image
!pip -q install -r requirements.txt

# --- Checagens antes de gastar tempo ---
import os, torch
assert torch.cuda.is_available(), 'SEM GPU. Ambiente de execução -> Alterar tipo -> T4 GPU e rode de novo.'
assert os.environ.get('DATA_DIR'), 'Rode a célula #0 antes (ela define DATA_DIR).'
print('GPU:', torch.cuda.get_device_name(0), '| DATA_DIR:', os.environ['DATA_DIR'])

# --- Autenticação via Colab Secrets (🔑 -> criar HF_TOKEN de ESCRITA -> habilitar Notebook access) ---
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

# --- Treinamento (usa o MESMO DATA_DIR da célula #0 via "$DATA_DIR") ---
# IMPORTANTE: --mixed_precision='fp16' precisa ir para o accelerate E para o script.
# Sem ele no script, os pesos do LoRA não são convertidos p/ fp32 e ocorre
# "Attempting to unscale FP16 gradients."
!accelerate launch \
  --num_processes=1 --num_machines=1 \
  --mixed_precision='fp16' --dynamo_backend='no' \
  train_text_to_image_lora.py \
  --pretrained_model_name_or_path='stable-diffusion-v1-5/stable-diffusion-v1-5' \
  --train_data_dir="$DATA_DIR" \
  --caption_column='text' \
  --mixed_precision='fp16' \
  --resolution=512 --random_flip \
  --train_batch_size=1 --gradient_accumulation_steps=4 \
  --max_train_steps=1500 --learning_rate=1e-4 --lr_scheduler='cosine' \
  --lr_warmup_steps=0 --rank=8 \
  --checkpointing_steps=500 --seed=42 \
  --output_dir='/content/atelie-lora' \
  --validation_prompt='estilo_lambelambe, retrato de uma feira de domingo' \
  --push_to_hub --hub_model_id='lamble-lambe/atelie'